<a href="https://colab.research.google.com/github/1stzl01/Reserving/blob/main/ICR_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Micro-Level Stochastic Claims Reserving (ICR Framework)

> **Methodology & Attribution:**
> The micro-level simulation engine implemented in this notebook is based on the **Individual Claim Reserving (ICR)** method.
>
> *Reference:* Chalnot, J.B., Gremillet, M., & Miehe, P. (2015). *Implementing the Individual Claims Reserving Method, a new approach in non-life reserving*. ASTIN Bulletin.

This notebook implements the **Individual Claim Reserving (ICR)** methodology for non-life insurance. Unlike classical deterministic methods (e.g., Chain Ladder) that aggregate data into run-off triangles, this micro-level approach utilizes the exact timeline of every individual claim to eliminate aggregation bias.

**Methodology:**
1. **Data Preparation**: Process individual claim logs capturing occurrence, notification, payments, and closure.
2. **IBNR Frequency**: Estimate the number of Incurred But Not Reported (IBNR) claims using a Poisson distribution based on historical exposure.
3. **Transition Probabilities**: Calculate the empirical probabilities of 4 distinct events for open claims: Closure without payment (Type 1), Closure with payment (Type 2), Intermediate payment (Type 3), and an "Empty" event (Type 4).
4. **Severity Calibration**: Fit a Gamma distribution to historical payments to model future claim severities.
5. **Stochastic Simulation Engine**: Project every open claim (RBNS) and simulated IBNR forward period-by-period until closure to calculate the final reserve distribution.



In [12]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import gamma

np.random.seed(42)
num_claims = 10000

# 1. Simulate historical data
data = {
    'claim_id': [f"CLM_{i}" for i in range(num_claims)],
    'occurrence_year': np.random.randint(2010, 2023, size=num_claims),
    # Corrected array: reporting delay D_ik takes values 1, 2, 3, or 4
    'reporting_delay': np.random.choice((1, 2, 3, 4), p=[0.7, 0.2, 0.08, 0.02], size=num_claims),
    'status': np.random.choice(['Closed', 'Open'], p=[0.85, 0.15], size=num_claims)
}
df_claims = pd.DataFrame(data)

# The reporting year formula: i + D_ik - 1
df_claims['reporting_year'] = df_claims['occurrence_year'] + df_claims['reporting_delay'] - 1

# 2. Simulate payment history for these claims
payments = []
for _, row in df_claims.iterrows():
    dev_periods = np.random.randint(1, 5)
    for dev in range(1, dev_periods + 1):
        # Corrected array: Event Types 1, 2, 3, or 4
        event_type = np.random.choice((1, 2, 3, 4), p=[0.3, 0.4, 0.1, 0.2])

        # Corrected array: Payments only occur on event types 2 and 3
        #payment_amt = gamma.rvs(a=2, scale=1000) if event_type in [7, 8] else 0

        # CORRECTED: Use tuple (2, 3) instead of brackets
        payment_amt = gamma.rvs(a=2, scale=1000) if event_type in (2, 3) else 0


        payments.append({
            'claim_id': row['claim_id'],
            'dev_year': dev,
            'event_type': event_type,
            'payment_amt': payment_amt
        })

df_payments = pd.DataFrame(payments)

print(f"Loaded {len(df_claims)} claims and {len(df_payments)} historical payment events.")

Loaded 10000 claims and 25094 historical payment events.


In [13]:
# The ICR model estimates the frequency parameter (lambda_tilde) by dividing
# the historical number of declared claims by the exposure of the origin period.

# 1. Aggregate reported claims by occurrence year and reporting delay
ibnr_agg = df_claims.groupby(['occurrence_year', 'reporting_delay']).size().reset_index(name='claim_count')

# 2. Add realistic exposure data (e.g., number of policies written per year)
exposure_data = {year: 50000 + np.random.randint(-5000, 5000) for year in range(2010, 2024)}
ibnr_agg['exposure'] = ibnr_agg['occurrence_year'].map(exposure_data)

# 3. Fit a Poisson Generalized Linear Model (GLM) to find the reporting frequency per delay
# Model: claim_count ~ reporting_delay + offset(log(exposure))
poisson_model = sm.GLM(
    endog=ibnr_agg['claim_count'],
    exog=sm.add_constant(ibnr_agg['reporting_delay']),
    family=sm.families.Poisson(),
    offset=np.log(ibnr_agg['exposure'])
).fit()

# Extract parameters to calculate lambda_tilde for future simulations
print("Poisson GLM Calibration for IBNR Frequency:")

# Print the entire summary to avoid index errors
print(poisson_model.summary())


Poisson GLM Calibration for IBNR Frequency:
                 Generalized Linear Model Regression Results                  
Dep. Variable:            claim_count   No. Observations:                   52
Model:                            GLM   Df Residuals:                       50
Model Family:                 Poisson   Df Model:                            1
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -225.33
Date:                Thu, 23 Apr 2026   Deviance:                       120.17
Time:                        20:11:03   Pearson chi2:                     119.
No. Iterations:                     6   Pseudo R-squ. (CS):              1.000
Covariance Type:            nonrobust                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------

In [14]:
# Calculate the empirical probability of a future event being Type m (1, 2, 3, or 4)
# for a given development period j. Formula: P(E_j = m) = N_m(j) / sum(N_k(j)).

# Count events by development year and type
transition_counts = df_payments.groupby(['dev_year', 'event_type']).size().unstack(fill_value=0)

# Normalize by row to get probabilities
transition_probs = transition_counts.div(transition_counts.sum(axis=1), axis=0)
transition_probs_dict = transition_probs.to_dict(orient='index')

print("Calibrated 4-State Transition Probabilities by Development Year:")
print(transition_probs)

Calibrated 4-State Transition Probabilities by Development Year:
event_type         1         2         3         4
dev_year                                          
1           0.299800  0.396400  0.104500  0.199300
2           0.299643  0.396216  0.101998  0.202143
3           0.299000  0.404400  0.098000  0.198600
4           0.304142  0.391321  0.107298  0.197239


In [15]:
# Payments are modelled using a fine-tailed distribution (Gamma).
# We calibrate the Gamma shape (a) and scale parameters for each development year.

severity_params = {}
#paid_events = df_payments[df_payments['event_type'].isin([10, 11])]

# CORRECTED: Use tuple (2, 3) for the .isin() filter
paid_events = df_payments[df_payments['event_type'].isin((2, 3))]

for dev_year, group in paid_events.groupby('dev_year'):
    data = group['payment_amt']
    if len(data) > 10: # Ensure sufficient sample size
        # Scipy fit returns (shape, loc, scale)
        shape, loc, scale = gamma.fit(data, floc=0)
        severity_params[dev_year] = {'shape': shape, 'scale': scale}

print("Gamma Distribution Parameters Calibrated for Paid Events.")

Gamma Distribution Parameters Calibrated for Paid Events.


In [16]:
# Every open claim (RBNS) and simulated IBNR is iteratively projected period by period.
# If an event involves a payment (Type 2 or 3), the severity distribution is used.

# 1. Identify Open Claims (RBNS)
open_claims = df_claims[df_claims['status'] == 'Open'].copy()

# 2. Simulation Engine
total_simulated_reserve = 0
simulation_log = []

for index, claim in open_claims.iterrows():
    # Find current development year for the claim
    past_events = df_payments[df_payments['claim_id'] == claim['claim_id']]
    current_dev = past_events['dev_year'].max() + 1 if not past_events.empty else 1

    claim_closed = False

    # Project forward until closure or maximum development limit
    while not claim_closed and current_dev <= 11:
        # Get transition probabilities for current development year (default to last known if beyond)
        probs_row = transition_probs_dict.get(current_dev, transition_probs_dict[max(transition_probs_dict.keys())])
        probs = [probs_row.get(1, 0), probs_row.get(2, 0), probs_row.get(3, 0), probs_row.get(4, 0)]


        # Simulate Event
        event = np.random.choice((1, 2, 3, 4), p=probs)

        payment = 0

        # CORRECTED: Use tuple (2, 3). Types 2 and 3 trigger a payment simulation
        if event in (2, 3):
            params = severity_params.get(current_dev, severity_params[max(severity_params.keys())])
            payment = gamma.rvs(a=params['shape'], scale=params['scale'])
            total_simulated_reserve += payment

        # CORRECTED: Use tuple (1, 2). Types 1 and 2 mean the claim is closed
        if event in (1, 2):
            claim_closed = True

        simulation_log.append({
            'claim_id': claim['claim_id'],
            'simulated_dev_year': current_dev,
            'event_type': event,
            'simulated_payment': payment
        })

        current_dev += 1

print(f"--- MICRO-LEVEL ICR STOCHASTIC RESERVING COMPLETE ---")
print(f"Total simulated reserve (RBNS only) for {len(open_claims)} open claims: €{total_simulated_reserve:,.2f}")

# Display a sample of the individualized projection paths
sim_df = pd.DataFrame(simulation_log)
sim_df.head(10)

--- MICRO-LEVEL ICR STOCHASTIC RESERVING COMPLETE ---
Total simulated reserve (RBNS only) for 1544 open claims: €2,228,739.75


,claim_id,simulated_dev_year,event_type,simulated_payment
0,CLM_0,2,2,2670.247740
1,CLM_10,4,2,1279.031639
2,CLM_12,2,3,480.320719
3,CLM_12,3,3,4714.362821
4,CLM_12,4,1,0.000000
5,CLM_15,5,1,0.000000
6,CLM_16,2,2,264.407660
7,CLM_22,5,2,1842.393348
8,CLM_26,2,4,0.000000
9,CLM_26,3,1,0.000000
